# Scaling and operating in production

**Optional depth track · module 5 of 5**

**Goal:** Emit one log line you would still want at 3am, measure your own latency instead of guessing it, and write the page somebody reads when it breaks.

**Why it matters:** Operating is where a number stops being a hope. The habit worth building is labelling every figure as measured or target, because the two look identical in a document and only one of them is true.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## 1. One log line

**Context.** Logs get shipped, copied into tickets and kept far longer than anyone intends.
Put in what lets you follow one call through the system. Leave out everything else.

**Instructions.**

1. Emit one structured line as a dict: at minimum `request_id`, `event`, `duration_ms`.
2. The unit goes in the field name. `duration: 412` is unreadable in six months.
3. The check refuses a line containing the question, the answer, or anything key-shaped.

In [ ]:
line = {
    "request_id": "req_01J7E8Q2M4",
    "event": "answer_completed",
    # TODO(you): how long it took, and whatever else you would want at 3am.
    # Nothing that identifies a person or repeats their question.
}
print(line)

**Expected output**

```
{'request_id': 'req_01J7E8Q2M4', 'event': 'answer_completed', 'duration_ms': 412, ...}
✅ d5-e1 passed
```

In [ ]:
check("d5-e1", line)

## 2. Measure it, do not estimate it

**Context.** Run the agent enough times to have a distribution, then report p50 and p95 from
your own machine. The number matters less than the habit of labelling where it came from.

**Instructions.**

1. Run at least 20 iterations. A percentile over five samples is not a percentile.
2. Compute p50 and p95 in milliseconds.
3. Set `source` to `"measured"`. The check refuses `"target"`, because a goal is not a result.

In [ ]:
import statistics
import time

from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.agent import answer_question

documents = load_corpus(REPO_ROOT / "data" / "corpus")
timings = []
for _ in range(25):
    started = time.perf_counter()
    answer_question("What is chunking?", documents, FakeLLM())
    timings.append((time.perf_counter() - started) * 1000)

measurement = {
    "p50_ms": round(statistics.median(timings), 2),
    "p95_ms": 0.0,        # TODO(you): the 95th percentile of `timings`
    "runs": len(timings),
    "source": "",         # TODO(you): measured or target?
}
print(measurement)

**Expected output** (your numbers will differ; the shape will not):

```
{'p50_ms': 0.42, 'p95_ms': 0.71, 'runs': 25, 'source': 'measured'}
✅ d5-e2 passed
```

In [ ]:
check("d5-e2", measurement)

## 3. The runbook

**Context.** Written for somebody who did not build this, reading it while something is
broken. Four lines. The last one is the one people forget, and it is the one that tells you
whether you fixed it or just waited.

**Instructions.**

1. `symptom` is what a person notices, in their words, not in yours.
2. `first_check` is the single command or dashboard to open first.
3. `rollback` is the exact way back to the last good state.
4. `how_you_know` is the observation that proves recovery.

In [ ]:
runbook = {
    "symptom": "Answers come back refused for questions the corpus definitely covers.",
    "first_check": "",   # TODO(you): the one thing to look at first
    "rollback": "",      # TODO(you): the exact way back
    "how_you_know": "",  # TODO(you): how you know it worked
}
for k, v in runbook.items():
    print(f"{k:14} {v or '(empty)'}")

**Expected output**

```
symptom        Answers come back refused for questions the corpus definitely covers.
first_check    Run the eight golden cases. ...
rollback       Redeploy the previous commit and re-ingest ...
how_you_know   The golden set returns to three refusals out of eight, ...
✅ d5-e3 passed
```

In [ ]:
check("d5-e3", runbook)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d5")